In [0]:
%pip install laspy[lazrs,laszip] geopandas
%restart_python

File: https://www.land.vic.gov.au/maps-and-spatial/imagery/elevation-data/major-lidar-projects/digital-twin-victoria-lidar-2021-24

In [0]:
import laspy

with laspy.open(
    '/Volumes/danny_catalog/demo_schema/raw_data/e475n5952_myrtleford_2022jan21_mpts-c2_v10cm_ahd_epsg7855.copc.laz', 
    laz_backend=laspy.LazBackend.Laszip
) as fh:
    las = fh.read()

In [0]:
import numpy as np
import pandas as pd
from shapely.geometry import Point

# Limit to first 10 records
n = 10
x = las.x[:n]
y = las.y[:n]
z = las.z[:n]
intensity = [int(i) for i in las.intensity[:n]]
classification = [int(c) for c in las.classification[:n]]
return_number = [int(rn) for rn in las.return_number[:n]]

# Create points from x, y, z coordinates
geometry = [Point(xyz) for xyz in zip(x, y, z)]

# Convert geometry to WKT text
geometry_wkt = [geom.wkt for geom in geometry]

# Create a DataFrame with point attributes and WKT geometry
lidar_df = pd.DataFrame({
    'intensity': intensity,
    'classification': classification,
    'return_number': return_number,
    'geometry_wkt': geometry_wkt  # Store geometry as WKT text
})

In [0]:
display(lidar_df)

In [0]:
df = spark.createDataFrame(lidar_df)
display(df)

In [0]:
(
  df.write
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("danny_catalog.demo_schema.lidar_bronze")
)

Coordinate reference system: GDA2020 MGA 54/55, AHD - EPSG 7854

In [0]:
%sql
CREATE OR REPLACE TABLE danny_catalog.demo_schema.lidar_silver AS
SELECT 
  intensity, 
  classification, 
  return_number, 
  geometry_wkt,
  st_x(st_transform(st_setsrid(st_geomfromwkt(geometry_wkt), 7854), 4326)) as x_4326,
  st_y(st_transform(st_setsrid(st_geomfromwkt(geometry_wkt), 7854), 4326)) as y_4326,
  st_z(st_transform(st_setsrid(st_geomfromwkt(geometry_wkt), 7854), 4326)) as z_4326
FROM danny_catalog.demo_schema.lidar_bronze;

SELECT * FROM danny_catalog.demo_schema.lidar_silver

Databricks visualization. Run in Databricks to view.